# Graph Summarization Algorithm

This notebook demonstrates a simplified DAG-based graph summarization algorithm applied to a CFG tha should be effectively the same as summarize_graph.py.

For simplicity, this doesn't apply the existing pruning step, but it should work with or without.

## Algorithm Overview

1. Initialize empty summaries for all functions/communities
2. Use GreedyFAS to remove the minimum number of edges to create an acyclic graph (DAG) (using `nx.minimum_feedback_arc_set`), which is required to compute a topological sort.
3. Compute reverse topological sort to determine processing order (using `nx.topological_sort`) 
4. Process functions sequentially. The topological sort step (3) guarantees that dependencies are summarized first, so they can be used as context for subsequent summarizations.


In [1]:
import networkx as nx
import matplotlib.pyplot as plt
from ipysigma import Sigma
import numpy as np
from collections import defaultdict

## 0. Load CFG

In [4]:
from src.visualize_cfg import load_cfg

G = load_cfg('reorder_and_pad2.json')

## 1. Detect and Aggregate Communities

For now I'm just using the Leiden Method on the individual block nodes, creating more even sized units to be summarized, although they may not correspond to logical units as well as just using functions. We can investigate later if its better to start with entire functions or not.

In [5]:
from src.community_detection import collapse_leiden
G_communities, community_list = collapse_leiden(G, resolution=0.001)

widget = Sigma(
    G_communities,
    node_label='label',
    raw_node_color=lambda n, d: "red" if "main" in d.get("func") else d.get('color'),
    edge_color='type',
    label_font='sans-serif',
    default_edge_type='arrow',
    start_layout=10,
)
display(widget)

Sigma(nx.DiGraph with 173 nodes and 461 edges)

## 2. Remove Cycles using GreedyFAS

The minimum feedback arc set problem (FAS) is NP-hard so tihs is a vibe-coded implementation of GreedyFAS ([paper](https://arxiv.org/pdf/2208.09234)), an approximate version. It reomves the minimum number of edges in a directed graph to make it acyclic.



In [6]:
import networkx as nx

def greedy_feedback_arc_set(G):
    feedback_edges = set()
    visited = set()
    rec_stack = set()
    
    def dfs(node):
        visited.add(node)
        rec_stack.add(node)
        
        for neighbor in G.successors(node):
            if neighbor not in visited:
                dfs(neighbor)
            elif neighbor in rec_stack:
                # Back edge found - add to feedback arc set
                feedback_edges.add((node, neighbor))
        
        rec_stack.remove(node)
    
    for node in G.nodes():
        if node not in visited:
            dfs(node)
    
    return feedback_edges

# Apply greedy FAS
fas_edges = greedy_feedback_arc_set(G_communities)

# Create DAG by removing FAS edges
G_dag = G_communities.copy()
G_dag.remove_edges_from(fas_edges)

widget = Sigma(
    G_dag,
    node_label='label',
    raw_node_color=lambda n, d: "red" if "main" in d.get("func") else d.get('color'),
    edge_color='type',
    label_font='sans-serif',
    default_edge_type='arrow',
    start_layout=10,
)
display(widget)

print(f"FAS size: {len(fas_edges)} edges")
print(f"Before FAS removal: {G_communities.number_of_nodes()} nodes, {G_communities.number_of_edges()} edges")
print(f"After FAS removal: {G_dag.number_of_nodes()} nodes, {G_dag.number_of_edges()} edges")
    

Sigma(nx.DiGraph with 173 nodes and 316 edges)

FAS size: 145 edges
Before FAS removal: 173 nodes, 461 edges
After FAS removal: 173 nodes, 316 edges


## 3. Compute Reverse Topological Sort

Now that the graph is a DAG, we can find a topological sort and simply summarize them in reverse order. This guarantees that every node's dependencies/children are summarized before the node itself, without any tricky graph traversal stuff (e.g. checking if a node has been summarized).

In [7]:
summary_order = list(reversed(list(nx.topological_sort(G_dag))))
print(summary_order)
print(len(summary_order))

[31, 72, 145, 17, 86, 75, 20, 9, 38, 93, 65, 127, 101, 51, 83, 43, 25, 12, 23, 141, 11, 19, 124, 55, 105, 41, 63, 58, 16, 77, 78, 122, 69, 6, 14, 150, 117, 100, 56, 45, 87, 30, 42, 138, 129, 27, 104, 35, 54, 119, 108, 66, 62, 130, 139, 4, 95, 13, 34, 10, 59, 118, 90, 84, 112, 142, 57, 28, 102, 3, 2, 21, 48, 140, 120, 88, 44, 80, 5, 49, 8, 33, 1, 111, 113, 68, 61, 110, 92, 147, 39, 74, 15, 85, 155, 73, 64, 26, 71, 46, 40, 99, 133, 36, 96, 114, 132, 37, 70, 107, 22, 67, 79, 29, 76, 53, 157, 156, 47, 24, 89, 135, 109, 137, 7, 18, 52, 50, 103, 60, 81, 172, 171, 170, 169, 168, 167, 166, 165, 164, 163, 162, 161, 160, 159, 158, 154, 153, 152, 151, 149, 148, 146, 144, 143, 136, 134, 131, 128, 126, 125, 123, 121, 116, 115, 106, 98, 97, 94, 91, 82, 32, 0]
173


## 4. Summarize with LLM in Reverse Topological Order

Now we can simply loop through the list of nodes sequentially, and know that for each node we will have as many dependencies summarized as possible by that iteration. We add the LLM output to the summary attribute of the **original** graph with all original control flow edges, and save it at the end.

*Ideally, we would save as we go, or do some kind of checkpointing, but this notebook is just for demonstrating the idea anyway.*

In [ ]:
import ollama
from tqdm import tqdm
import json

# System prompts from summarize_graph.py
FUNC_SYSTEM_PROMPT = (
    "You are a reverse-engineering assistant. "
    "You are given all basic blocks of a single function's aarch64 assembly, "
    "along with the control flow between them. "
    "Summarize what this function does. Be concise. "
    "Your audience is an expert reverse engineer. "
    "If a function follows standard ABI calling conventions don't reexplain them."
)

FUNC_WITH_DEPS_SYSTEM_PROMPT = (
    "You are a reverse-engineering assistant. "
    "You are given all basic blocks of a single function's aarch64 assembly, "
    "along with the control flow between them, followed by summaries of "
    "other functions that this function calls or jumps to. "
    "Summarize what this function does at a high level, incorporating "
    "what the called/referenced functions do. Be concise. "
    "If a summary is not yet available for a called/referenced function, just note how that funtion is called or jumped to. "
    "Focus on how inputs are used and transformed into outputs. "
    "Your audience is an expert reverse engineer. You need to provide them"
    " an accurate understanding of the function's behavior. "
    "If a function follows standard ABI calling conventions don't reexplain them."
)

# Configure LLM
model = "qwen2.5-coder:7b" 
base_url = "http://localhost:11434"
client = ollama.Client(host=base_url)

# Process all nodes
for node in tqdm(summary_order):
    successors = list(G_dag.successors(node))
    
    # Build dependency summaries
    deps_text = ""
    for succ in successors:
        succ_summary = G_communities.nodes[succ].get('summary', "")
        if succ_summary:
            deps_text += f"\n--- Called/Referenced function: {succ} ---\n{succ_summary}\n"
    
    # Construct the task prompt based on whether dependencies are available
    if deps_text:
        user_content = (
            f"=== Function: {node} ===\n"
            f"=== Called/Referenced Function Summaries ==={deps_text}"
        )
        system_prompt = FUNC_WITH_DEPS_SYSTEM_PROMPT
    else:
        user_content = f"=== Function: {node} ==="
        system_prompt = FUNC_SYSTEM_PROMPT
    
    response = client.generate(model=model, prompt=f"{system_prompt}\n\n{user_content}", stream=False)
    summary = response["response"]
    G_communities.nodes[node]['summary'] = summary

# Save summaries
summaries = {node: G_communities.nodes[node]['summary'] for node in G_communities.nodes() if 'summary' in G_communities.nodes[node]}
with open('summaries.json', 'w') as f:
    json.dump(summaries, f, indent=2)
print(f"Summarization complete. Saved {len(summaries)} summaries to summaries.json")

  0%|          | 0/173 [00:00<?, ?it/s]

100%|██████████| 173/173 [06:28<00:00,  2.25s/it]

Summarization complete. Saved 173 summaries to summaries.json


## Hierarchal Approach

This could be applied in a hierarchal way, but detecting and aggregating communities may re-introduce cycles, so the GreedyFAS edge removal step will have to be applied at each level.